# Part 1. Equation of a Slime

How many late days are you using for this assignment? 
Answer: 0

In [ ]:
# Imports section
# Used in both parts
import pandas as pd
import numpy as np
import sklearn

# Part 1
from sklearn import model_selection, linear_model, preprocessing, metrics, neighbors, neural_network, svm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures

# Part 2
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

## 1. Loading the dataset

In [7]:
# Using pandas load the dataset
data = pd.read_csv("science_data_large.csv")

# Output the first 15 rows of the data
print(data.head(15))

# Display a summary of the table information (number of datapoints, etc.)
print(data.info())


    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## 2. Splitting the dataset

In [11]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = data[["Temperature °C", "Mols KCL"]]  # input features
y = data["Size nm^3"]  # output label

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Training set size: 900 samples
Test set size: 100 samples


## 3. Perform a Linear Regression

In [47]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)


# Create a sample datapoint and predict the output of that sample with the trained model
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
sample = [[300, 300]]
prediction = model.predict(sample)
print(f"Predicted Size (nm^3) for input {sample}: {prediction[0]:.5f}")


# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print(f"Model Score (R²): {score:.5f}")


# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_ 
intercept = model.intercept_
print(f"Intercept: {intercept:.5f}")
print(f"Coefficients: {coefficients}")

Predicted Size (nm^3) for input [[300, 300]]: 160260.96437
Model Score (R²): 0.85525
Intercept: -409391.47958
Coefficients: [ 866.14641337 1032.69506649]


Write the linear equation of a slime: (example equation: $E = mc^2$):
$$
h(x) = -409391.47958 + (886.14641 \times \mathrm{Temperature}^\circ\mathrm{C}) + (1032.69507 \times \mathrm{Mols \, KCL})
$$

Report on score and explain meaning:
A score of 0.85525 indicates that our model is performing well. This means it can accurately predict the size (nm^3) based on inputs of Temperature (°C) and Mols of KCL. Model scores range from 0 to 1, where 0 represents a completely ineffective model that is no better than random guessing. Scores closer to 1, like our 0.85525, indicate a more reliable model that makes accurate predictions. A score of 1 would mean perfect predictions, where the model consistently produces correct outputs based on given inputs. In rare cases, negative scores can occur, which would indicate that the model is performing worse than random guessing and failing to capture any meaningful pattern in the data. Overall, our score suggests a strong but not perfect model, making it a useful tool for predicting outcomes based on the provided features.


## 4. Use Cross Validation

In [52]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring='r2')
print("Cross-validation scores (R²):", scores)
print(f"Mean R²: {scores.mean():.5f}")
print(f"Standard Deviation: {scores.std():.5f}")

# Report on their finding and their significance

Cross-validation scores (R²): [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
Mean R²: 0.85973
Standard Deviation: 0.01839


Write findings here: In this experiment, we use KFold to shuffle the dataset five times. This method is a form of cross-validation, where we validate our initial experiment score by comparing it with scores obtained from multiple shuffles of the same dataset. Each shuffle splits the data into five folds, with one fold used for testing and the remaining four for training. This process is repeated across all five shuffles, ensuring that different parts of the data serve as the test set. As a result, we obtain five separate scores—one for each iteration—and then average them to produce a final score. In our case, the final score is 0.85973, which is very close to our earlier single-split score of 0.85525. This consistency suggests that our initial score was not a fluke, which makes sense given our relatively large dataset of 1,000 entries.

## 5. Using Polynomial Regression

In [60]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)  

# Perform k-fold cross validation (as above)
model_poly = LinearRegression()
kfold_poly = KFold(n_splits=5, shuffle=True, random_state=42)
scores_poly = cross_val_score(model_poly, X_poly, y, cv=kfold_poly, scoring='r2')
print("Cross-validation scores (R²):", scores_poly)
print("Mean R²:", scores_poly.mean())
print("Standard Deviation:", scores_poly.std())

# Report on the metrics and output the resultant equation as you did in Part 3.
model_poly.fit(X_poly, y)
coefficients_poly = model_poly.coef_
intercept_poly = model_poly.intercept_
print(f"Intercept: {intercept_poly:.5f}")
print(f"Coefficients: {coefficients_poly}")
feature_names = poly.get_feature_names_out(input_features=["Temperature °C", "Mols KCL"])
print(feature_names)

Cross-validation scores (R²): [1. 1. 1. 1. 1.]
Mean R²: 1.0
Standard Deviation: 0.0
Intercept: 0.00002
Coefficients: [ 1.20000000e+01 -1.23111736e-07 -1.05952996e-11  2.00000000e+00
  2.85714287e-02]
['Temperature °C' 'Mols KCL' 'Temperature °C^2' 'Temperature °C Mols KCL'
 'Mols KCL^2']


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$$
\begin{align}
h(\mathrm{Temperature}, \mathrm{Mols\ KCL}) &= 0.00002\\
&+ (1.2 \times 10^{1} \cdot \mathrm{Temperature})\\
&- (1.23111 \times 10^{-7} \cdot \mathrm{Mols\ KCL})\\
&- (1.05953 \times 10^{-11} \cdot \mathrm{Temperature}^2)\\
&+ (2.0 \times 10^{0} \cdot (\mathrm{Temperature} \cdot \mathrm{Mols\ KCL}))\\
&+ (2.85714 \times 10^{-2} \cdot \mathrm{Mols\ KCL}^2)
\end{align}
$$

Report on the score and interpret:
Polynomial regression performed exceptionally well in modeling the relationship between inputs and outputs. Each of the five cross-validation tests returned a perfect score of 1.0, indicating that the model could perfectly predict the size of the slime based on the temperature and the number of moles of KCL it was reacted with. This improvement over the linear regression model suggests two key insights about our dataset:

The relationship is non-linear.
Just as polynomial equations create curved lines when plotted, our dataset follows a pattern that cannot be represented by a single straight line. This makes sense when we reconsider the intercept from the linear regression model. The linear model produced an intercept of -409391.47958, which is unrealistic—it suggests that the slime would have a negative size at low temperatures and small quantities of KCL. This flaw is corrected in the polynomial regression model, where the intercept is 0.00002, ensuring that size only increases as temperature and Mols KCL increase.

Our dataset is highly predictable.
With linear regression, we could reliably predict outcomes, but not perfectly. However, since the polynomial model achieved a perfect score (R² = 1.0) on all cross-validation tests, we can conclude that it can perfectly predict outcomes for any given input. This is further supported by the more complex polynomial equation, which accounts for nonlinear interactions by incorporating squared terms and interaction terms. Unlike the linear model, which only used two features, the polynomial model captures a more accurate and detailed representation of how the inputs influence the output.

# Part 2. Chronic Kidney Disease Prediction via Classification

# Load ckd_featuer_subset.csv

In [64]:
# Load the dataset. Then train and evaluate the classification models.
cdk_data = pd.read_csv("ckd_feature_subset.csv")
X = cdk_data.drop("Target_ckd", axis=1)
y = cdk_data["Target_ckd"]
print("Features (X):")
print(X.head())
print("\nTarget (y):")
print(y.head())

# Set random seed and create a KFold object
seed = 42
kfold = KFold(n_splits=5, shuffle=True, random_state=seed)

Features (X):
        age        bp      wbcc  appet_poor  appet_good      rbcc
0  0.688312  0.333333  0.000000           1           0  0.000000
1  0.545455  0.333333  0.128319           1           0  0.305085
2  0.714286  0.500000  0.238938           1           0  0.186441
3  0.688312  0.333333  0.283186           0           1  0.338983
4  0.441558  0.333333  0.221239           1           0  0.220339

Target (y):
0    1
1    1
2    1
3    1
4    1
Name: Target_ckd, dtype: int64


# Train & Evaluate Classification Models

In [68]:
# Define the classification models to evaluate
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=seed),
    "Support Vector Machines": SVC(random_state=seed),
    "k-Nearest Neighbors": KNeighborsClassifier(),
    "Neural Network": MLPClassifier(max_iter=1000, random_state=seed)
}

# Dictionary to store cross-validation results
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")
    results[name] = {"Mean score": np.mean(scores), "Std Dev": np.std(scores)}

# Create a DataFrame for a neat table of results
results_df = pd.DataFrame(results).T
print("Cross-validation results for classification models:")
print(results_df)


Cross-validation results for classification models:
                         Mean score   Std Dev
Logistic Regression        0.856559  0.066269
Support Vector Machines    0.928172  0.047601
k-Nearest Neighbors        0.927957  0.052440
Neural Network             0.935054  0.040466


Analysis of Classification Models:

The table above shows the cross-validation results for four classification models using 5-fold cross-validation. Each model's performance is reported as the mean accuracy and standard deviation across the folds, which gives us an idea of both the average performance and its consistency across different splits of the data. Based on these results, the Neural Network model performed the best, achieving the highest average accuracy with the lowest standard deviation. This suggests that the neural network was not only the most accurate on average, but its performance was also consistent across different subsets of the data. Neural networks are highly flexible and capable of modeling complex, non-linear relationships, which is likely why it outperformed the other models on this dataset. Logistic Regression, on the other hand, proved to be the least reliable. It produced the lowest average score along with the highest standard deviation. This is not unexpected, as Logistic Regression assumes a linear relationship between the inputs and the log-odds of the outputs. In real-world scenarios, data often contain non-linear patterns and interactions that a simple linear model cannot capture. As a result, Logistic Regression may oversimplify the underlying relationships, leading to lower accuracy and greater variability in its predictions. The other two models—Support Vector Machines (SVC) and k-Nearest Neighbors (k-NN)—performed similarly, with scores ranging between 0.9279 and 0.9282. Their comparable performance suggests that they are both well-suited to capture the non-linear structure in the data, although they might be more sensitive to noise compared to neural networks.

# Train & Evaluate Neural Networks

In [72]:
# Define multiple configurations for the MLPClassifier
nn_configs = {
    "Default": MLPClassifier(max_iter=1000, random_state=seed),
    "Hidden Layers (50,50), ReLU": MLPClassifier(hidden_layer_sizes=(50,50), max_iter=1000, activation="relu", random_state=seed),
    "Hidden Layer (100), tanh": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, activation="tanh", random_state=seed),
    "Hidden Layers (100,100), ReLU": MLPClassifier(hidden_layer_sizes=(100,100), max_iter=1000, activation="relu", random_state=seed),
    "Hidden Layer (50), logistic": MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, activation="logistic", random_state=seed),
    "Hidden Layers (75,50,25), ReLU": MLPClassifier(hidden_layer_sizes=(75,50,25), max_iter=1000, activation="relu", random_state=seed),
    "Hidden Layer (100), ReLU, adam": MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, activation="relu", solver="adam", random_state=seed),
    "Hidden Layers (50,50), tanh": MLPClassifier(hidden_layer_sizes=(50,50), max_iter=1000, activation="tanh", random_state=seed)
}

# Dictionary to store neural network results
nn_results = {}

for config_name, nn_model in nn_configs.items():
    nn_scores = cross_val_score(nn_model, X, y, cv=kfold, scoring="accuracy")
    nn_results[config_name] = {"Mean Accuracy": np.mean(nn_scores), "Std Dev": np.std(nn_scores)}

# Create a DataFrame for neural network results
nn_results_df = pd.DataFrame(nn_results).T
print("Neural Network configurations cross-validation results:")
print(nn_results_df)

Neural Network configurations cross-validation results:
                                Mean Accuracy   Std Dev
Default                              0.935054  0.040466
Hidden Layers (50,50), ReLU          0.954624  0.043696
Hidden Layer (100), tanh             0.941505  0.037327
Hidden Layers (100,100), ReLU        0.967527  0.035340
Hidden Layer (50), logistic          0.908817  0.051467
Hidden Layers (75,50,25), ReLU       0.967527  0.035340
Hidden Layer (100), ReLU, adam       0.935054  0.040466
Hidden Layers (50,50), tanh          0.935054  0.040466


Neural Network Experiment Findings:

The results show clear differences in performance among the various neural network configurations. The configurations with two hidden layers using ReLU—namely, "Hidden Layers (100,100), ReLU" and "Hidden Layers (75,50,25), ReLU"—achieved the highest mean accuracy of approximately 0.9675 with relatively low standard deviations (around 0.0353). This suggests that deeper architectures with multiple hidden layers and ReLU activation are very effective at capturing the complex patterns in the data. In contrast, the "Hidden Layer (50), logistic" configuration performed the worst, with a mean accuracy of about 0.9088 and the highest standard deviation (approximately 0.0515). The lower performance could be due to the logistic activation function's tendency to suffer from vanishing gradients, especially in networks that may require capturing more complex relationships. The default configuration and the "Hidden Layer (100), ReLU, adam" setup both achieved a mean accuracy of about 0.935, which is noticeably lower than the top performers. Similarly, the "Hidden Layers (50,50), tanh" configuration performed at a similar level to these models, indicating that while tanh can be effective in some cases, ReLU seems to provide a superior performance on this dataset. Overall, the results suggest that increasing the network depth and using ReLU as the activation function significantly boosts accuracy. The worst performance from the logistic activation underscores the importance of choosing an appropriate activation function for capturing non-linear patterns in the data.

# Final Data Table
| Configuration                       | Mean Accuracy | Std Dev  |
|-------------------------------------|---------------|----------|
| Logistic Regression                 | 0.856559      | 0.066269 |
| SVC                                 | 0.928172      | 0.047601 |
| k-NN                                | 0.927957      | 0.052440 |
| Neural Network (default)            | 0.935054      | 0.040466 |
| Hidden Layers (50,50), ReLU         | 0.954624      | 0.043696 |
| Hidden Layer (100), tanh            | 0.941505      | 0.037327 |
| Hidden Layers (100,100), ReLU       | 0.967527      | 0.035340 |
| Hidden Layer (50), logistic         | 0.908817      | 0.051467 |
| Hidden Layers (75,50,25), ReLU      | 0.967527      | 0.035340 |
| Hidden Layer (100), ReLU, adam      | 0.935054      | 0.040466 |
| Hidden Layers (50,50), tanh         | 0.935054      | 0.040466 |